# Conversational Memory

- Conversational memory allows our chatbots and agents to remember previous interactions within a conversation.

- Without conversational memory, our chatbots would only ever be able to respond to the last message they received, essentially forgetting all previous messages with each new message.

- Naturally, conversations require our chatbots to be able to respond over multiple interactions and refer to previous messages to understand the context of the conversation.


In [12]:
import os
import warnings
from pathlib import Path

# Standard imports
import numpy as np
import pandas as pd
import polars as pl
from typing import Any, Generator, Type, TypeVar

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

The lab_black extension is already loaded. To reload it, use:
  %reload_ext lab_black
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


<br>

## LangChain's Memory Types

- LangChain versions 0.0.x consisted of various conversational memory types.

- Most of these are due for deprecation but still hold value in understanding the different approaches that we can take to building conversational memory.

- Throughout the notebook we will be referring to these older memory types and then rewriting them using the recommended RunnableWithMessageHistory class. 

- We will learn about:
    - **ConversationBufferMemory**: the simplest and most intuitive form of conversational memory, keeping track of a conversation without any additional bells and whistles.
    
    - **ConversationBufferWindowMemory**: similar to ConversationBufferMemory, but only keeps track of the last k messages.
    
    - **ConversationSummaryMemory**: rather than keeping track of the entire conversation, this memory type keeps track of a summary of the conversation.
    
    - **ConversationSummaryBufferMemory**: merges the ConversationSummaryMemory and ConversationTokenBufferMemory types.

- We'll work through each of these memory types in turn, and rewrite each one using the RunnableWithMessageHistory class.

In [4]:
from langchain_openai import ChatOpenAI

model_str: str = "google/gemini-2.0-flash-001"

# Deterministic responses
llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str,
)

# Creative responses
creative_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),
    base_url=settings.OPENROUTER_URL,
    temperature=0.9,
    model=model_str,
)

### 1. ConversationBufferMemory

- ConversationBufferMemory is the simplest form of conversational memory, it is literally just a place that we store messages, and then use to feed messages into our LLM.

- Let's start with LangChain's original ConversationBufferMemory object, we are setting `return_messages=True` to return the messages as a list of ChatMessage objects — unless using a non-chat model we would always set this to True as without it the messages are passed as a direct string which can lead to unexpected behavior from chat LLMs.

In [5]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_72488/1448044083.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True)


- There are several ways that we can add messages to our memory, using the save_context method we can add a user query (via the input key) and the AI's response (via the output key). So, to create the following conversation:

    ```txt
    User: Hi, my name is Neidu
    AI: Hey Neidu, what's up? I'm an AI model called Zeta.

    User: I'm researching the different types of conversational memory.
    AI: That's interesting, what are some examples?

    User: I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.
    AI: That's interesting, what's the difference?

    User: Buffer memory just stores the entire conversation, right?
    AI: That makes sense, what about ConversationBufferWindowMemory?

    User: Buffer window memory stores the last k messages, dropping the rest.
    AI: Very cool!
    ```

We do:

In [ ]:
memory.save_context(
    {"input": "Hi, my name is Neidu"},  # user message
    {"output": "Hey Neidu, what's up? I'm an AI model called Zeta."},  # AI response
)
memory.save_context(
    {"input": "I'm researching the different types of conversational memory."},  # user message
    {"output": "That's interesting, what are some examples?"},  # AI response
)
memory.save_context(
    {"input": "I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory."},  # user message
    {"output": "That's interesting, what's the difference?"},  # AI response
)
memory.save_context(
    {"input": "Buffer memory just stores the entire conversation, right?"},  # user message
    {"output": "That makes sense, what about ConversationBufferWindowMemory?"},  # AI response
)
memory.save_context(
    {"input": "Buffer window memory stores the last k messages, dropping the rest."},  # user message
    {"output": "Very cool!"},  # AI response
)

- Before using the memory, we need to load in any variables for that memory type — in this case, there are none, so we just pass an empty dictionary:

In [7]:
memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey Neidu, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}),
  HumanMess

In [ ]:
memory = ConversationBufferMemory(return_messages=True)

memory.chat_memory.add_user_message("Hi, my name is Neidu")
memory.chat_memory.add_ai_message("Hey Neidu, what's up? I'm an AI model called Zeta.")
memory.chat_memory.add_user_message("I'm researching the different types of conversational memory.")
memory.chat_memory.add_ai_message("That's interesting, what are some examples?")
memory.chat_memory.add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
memory.chat_memory.add_ai_message("That's interesting, what's the difference?")
memory.chat_memory.add_user_message("Buffer memory just stores the entire conversation, right?")
memory.chat_memory.add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
memory.chat_memory.add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
memory.chat_memory.add_ai_message("Very cool!")

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey Neidu, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}),
  HumanMess

<br>

- The outcome is exactly the same in either case. To pass this onto our LLM, we need to create a `ConversationChain` object — which is already deprecated in favor of the `RunnableWithMessageHistory` class, which we will cover in a moment.

In [9]:
from langchain.chains import ConversationChain

chain = ConversationChain(llm=llm, memory=memory, verbose=True)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_72488/3211033272.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  chain = ConversationChain(llm=llm, memory=memory, verbose=True)


In [10]:
chain.invoke({"input": "What's my name again?"})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey Neidu, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}), HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kw

{'input': "What's my name again?",
 'history': [HumanMessage(content='Hi, my name is Neidu', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey Neidu, what's up? I'm an AI model called Zeta.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={},

<br>

### ConversationBufferMemory With RunnableWithMessageHistory

- As mentioned, the `ConversationBufferMemory` type is due for deprecation.

- Instead, we can use the `RunnableWithMessageHistory` class to implement the same functionality.

- When implementing `RunnableWithMessageHistory` we will use LangChain Expression Language `(LCEL)` and for this we need to define our prompt template and LLM components.

- Our llm has already been defined, so now we just define a ChatPromptTemplate object.

In [ ]:
from langchain.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
)

system_prompt: str = "You're a helpful assistant called Zeta."

prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(system_prompt),
        MessagesPlaceholder(variable_name="history"),
        HumanMessagePromptTemplate.from_template("{query}"),
    ]
)

# Connect the prompt_template with the LLM
pipeline = prompt_template | llm

- Our `RunnableWithMessageHistory` requires our pipeline to be wrapped in a `RunnableWithMessageHistory` object.

- This object requires a few input parameters and one of those is `get_session_history`, which requires a function that returns a ChatMessageHistory object based on a session ID.

- We define this function ourselves:

In [13]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_map: dict[str, Any] = {}


def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    """This is used to get the chat history."""
    if session_id not in chat_map:
        chat_map[session_id] = InMemoryChatMessageHistory()
    return chat_map[session_id]

- We also need to tell our runnable which variable name to use for the chat history (ie `history`) and which to use for the user's query (ie `query`).

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

pipeline_with_history = RunnableWithMessageHistory(
    runnable=pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
)

# Invoke the runnable
pipeline_with_history.invoke({"query": "Hi, my name is Neidu"}, config={"session_id": "id_123"})

AIMessage(content="Hi Neidu, it's nice to meet you! How can I help you today?\n", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 16, 'total_tokens': 36, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemini-2.0-flash-001', 'system_fingerprint': None, 'id': 'gen-1753995876-f8XKkTXdvBVef324OioG', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--564372b2-8b4d-4f06-b795-a732462dfd35-0', usage_metadata={'input_tokens': 16, 'output_tokens': 20, 'total_tokens': 36, 'input_token_details': {}, 'output_token_details': {}})

In [15]:
console.print(
    pipeline_with_history.invoke(
        {"query": "What's my name again?"},
        config={"session_id": "id_123"},
    )
)

AIMessage(
    content='Your name is Neidu.\n',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 7,
            'prompt_tokens': 42,
            'total_tokens': 49,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'google/gemini-2.0-flash-001',
        'system_fingerprint': None,
        'id': 'gen-1753995926-QMLhap5DCYK6z21KdPE1',
        'service_tier': None,
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='run--a60a9d93-73bd-4eec-bfca-549d21b98dac-0',
    usage_metadata={
        'input_tokens': 42,
        'output_tokens': 7,
        'total_tokens': 49,
        'input_token_details': {},
        'output_token_details': {}
    }
)

### 2. ConversationBufferWindowMemory

- The ConversationBufferWindowMemory type is similar to ConversationBufferMemory, but only keeps track of the `last k` messages. 

- There are a few reasons why we would want to keep only the last k messages:

- More messages mean more tokens are sent with each request, more tokens increases latency and cost.

- LLMs tend to perform worse when given more tokens, making them more likely to deviate from instructions, hallucinate, or "forget" information provided to them. Conciseness is key to high performing LLMs.

- If we keep all messages we will eventually hit the LLM's context window limit, by adding a window size k we can ensure we never hit this limit.

- The buffer window solves many problems that we encounter with the standard buffer memory, while still being a very simple and intuitive form of conversational memory.

In [16]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=4, return_messages=True)

In [ ]:
memory.chat_memory.add_user_message("Hi, my name is James")
memory.chat_memory.add_ai_message("Hey James, what's up? I'm an AI model called Zeta.")
memory.chat_memory.add_user_message("I'm researching the different types of conversational memory.")
memory.chat_memory.add_ai_message("That's interesting, what are some examples?")
memory.chat_memory.add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
memory.chat_memory.add_ai_message("That's interesting, what's the difference?")
memory.chat_memory.add_user_message("Buffer memory just stores the entire conversation, right?")
memory.chat_memory.add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
memory.chat_memory.add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
memory.chat_memory.add_ai_message("Very cool!")

memory.load_memory_variables({})